# Reddit Scraping 

In [1]:
import random
import re
import time
from typing import Dict, List
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd


In [6]:
reddit_urls = pd.read_csv('reddit_json_filled.csv')
reddit_urls = reddit_urls[['Reddit link', 'Home Team', 'Away Team']]
reddit_urls


,Reddit link,Home Team,Away Team
0,https://www.reddit.com/r/nfl/comments/1n8vmti/...,Philadelphia Eagles,Dallas Cowboys
1,https://www.reddit.com/r/nfl/comments/1nh64on/...,Kansas City Chiefs,Philadelphia Eagles
2,https://www.reddit.com/r/nfl/comments/1nn27zc/...,Philadelphia Eagles,Los Angeles Rams
3,https://www.reddit.com/r/nfl/comments/1nsym2c/...,Tampa Bay Buccaneers,Philadelphia Eagles
4,https://www.reddit.com/r/nfl/comments/1nyz09t/...,Philadelphia Eagles,Denver Broncos
...,...,...,...
165,https://www.reddit.com/r/nfl/comments/1ph3bzu/...,Kansas City Chiefs,Houston Texans
166,https://www.reddit.com/r/nfl/comments/1pmp8m4/...,Kansas City Chiefs,Los Angeles Chargers
167,https://www.reddit.com/r/nfl/comments/1psgz43/...,Tennessee Titans,Kansas City Chiefs
168,https://www.reddit.com/r/nfl/comments/1pvv072/...,Kansas City Chiefs,Denver Broncos


In [7]:
def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def jitter_sleep(min_s: float = 0.25, max_s: float = 0.8) -> None:
    time.sleep(random.uniform(min_s, max_s))

def to_old_reddit(url: str) -> str:
    if url.startswith("https://www.reddit.com"):
        return url.replace("https://www.reddit.com", "https://old.reddit.com")
    if url.startswith("http://www.reddit.com"):
        return url.replace("http://www.reddit.com", "https://old.reddit.com")
    if url.startswith("https://reddit.com"):
        return url.replace("https://reddit.com", "https://old.reddit.com")
    if url.startswith("http://reddit.com"):
        return url.replace("http://reddit.com", "https://old.reddit.com")
    return url

def with_query_params(url: str, **params: str) -> str:
    parsed = urlparse(url)
    q = dict(parse_qsl(parsed.query))
    for k, v in params.items():
        if v is None:
            q.pop(k, None)
        else:
            q[k] = str(v)
    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, urlencode(q), parsed.fragment))

def start_driver(headless: bool = False) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")  # Chrome 109+
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-notifications")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=opts)
    driver.set_page_load_timeout(30)
    return driver


In [18]:
MORECOMMENTS_SEL = "a.morecomments"

def click_more_comments(driver: webdriver.Chrome, max_clicks: int = 200) -> int:
    clicks = 0

    for _ in range(max_clicks):
        links = driver.find_elements(By.CSS_SELECTOR, MORECOMMENTS_SEL)

        candidates = []
        for a in links:
            try:
                if a.is_displayed():
                    txt = clean_text(a.text).lower()
                    if "more comments" in txt or "load more comments" in txt:
                        candidates.append(a)
            except Exception:
                continue

        if not candidates:
            break

        a = candidates[0]
        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", a)
            jitter_sleep(0.15, 0.35)
            a.click()
            clicks += 1
            jitter_sleep(0.6, 1.2)
        except Exception:
            jitter_sleep(0.4, 0.8)
            continue

    return clicks


In [19]:
def extract_comments_old(driver: webdriver.Chrome) -> List[Dict[str, str]]:
    results: List[Dict[str, str]] = []
    seen = set()

    comment_nodes = driver.find_elements(By.CSS_SELECTOR, "div.comment.thing")

    for c in comment_nodes:
        try:
            body_el = c.find_elements(By.CSS_SELECTOR, "div.entry div.md")
            if not body_el:
                continue

            text = clean_text(body_el[0].text)
            if not text:
                continue

            author_el = c.find_elements(By.CSS_SELECTOR, "a.author")
            author = clean_text(author_el[0].text) if author_el else ""

            key = (author, text)
            if key in seen:
                continue
            seen.add(key)

            results.append({"author": author, "text": text})
        except Exception:
            continue

    return results


In [20]:
def scrape_reddit_post_comments_notebook(
    post_url: str,
    sort: str = "new",
    max_clicks: int = 200,
    headless: bool = False,
    wait_seconds: int = 20,
) -> List[Dict[str, str]]:
    driver = start_driver(headless=headless)
    try:
        url = to_old_reddit(post_url)
        url = with_query_params(url, sort=sort)

        driver.get(url)
        WebDriverWait(driver, wait_seconds).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        jitter_sleep(1.0, 2.0)

        clicks = click_more_comments(driver, max_clicks=max_clicks)
        comments = extract_comments_old(driver)

        print(f"Clicked 'load more comments' {clicks} time(s). Extracted {len(comments)} comments.")
        return comments
    finally:
        return
        #driver.quit()


In [21]:
from __future__ import annotations

import argparse
import sys
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional

import requests

USER_AGENT = "proj (stevewon@andrew.cmu.edu)"
DEFAULT_TIMEOUT = 15

In [23]:
def http_get_json(url: str):
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/geo+json, application/json;q=0.9, */*;q=0.1",
    }
    try:
        resp = requests.get(url, headers=headers, timeout=DEFAULT_TIMEOUT)
    except requests.RequestException as e:
        raise ValueError(f"Network error while requesting {url}: {e}") from e

    if resp.status_code != 200:
        msg = f"API request failed ({resp.status_code}) for {url}"
        try:
            data = resp.json()
            detail = (
                data.get("detail")
                or data.get("title")
                or data.get("type")
                or str(data)[:200]
            )
            msg += f": {detail}"
        except ValueError:
            snippet = (resp.text or "").strip().replace("\n", " ")[:200]
            if snippet:
                msg += f": {snippet}"
        raise ValueError(msg)

    try:
        return resp.json()
    except ValueError as e:
        raise ValueError(f"Failed to decode JSON from {url}: {e}") from e

In [24]:
import csv
import json
import html
from typing import Any, Dict, List, Optional, Tuple


# --------------------------------
# JSON STRUCTURE HELPERS
# --------------------------------

def _extract_post_and_top_listing(thread_json: Any) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    thread_json is expected to be:
      [0] post Listing
      [1] comments Listing
    """
    post_listing = thread_json[0]
    comments_listing = thread_json[1]
    post = post_listing["data"]["children"][0]["data"]
    return post, comments_listing


def _clean_body(body: Any) -> Optional[str]:
    if body is None:
        return None
    # Unescape HTML entities like &amp; and &#39;
    return html.unescape(str(body))


def _normalize_edited(edited_val: Any) -> Optional[float]:
    """
    Reddit 'edited' is either False or a unix timestamp (float/int).
    We normalize to:
      - None if not edited
      - float timestamp if edited
    """
    if edited_val is False or edited_val is None or edited_val == "":
        return None
    try:
        return float(edited_val)
    except Exception:
        return None


def _normalize_distinguished(dist: Any) -> Optional[str]:
    """
    Normalize blanks/False to None; otherwise string.
    """
    if dist in ("", None, False):
        return None
    return str(dist)


def _safe_int(x: Any, default: int = 0) -> int:
    if x is None or x == "":
        return default
    try:
        return int(x)
    except Exception:
        try:
            return int(float(x))
        except Exception:
            return default


def _flatten_comment(
    c: Dict[str, Any],
    post: Dict[str, Any],
    depth: int,
) -> Dict[str, Any]:
    """
    Produces a row matching your CSV schema, with cleaned/normalized fields.
    """
    return {
        "post_id": post.get("id"),
        "link_fullname": post.get("name"),            # t3_xxx
        "comment_id": c.get("id"),
        "comment_fullname": c.get("name"),           # t1_xxx
        "parent_fullname": c.get("parent_id"),
        "depth": depth,
        "author": c.get("author"),
        "body": _clean_body(c.get("body")),
        "score": _safe_int(c.get("score"), default=0),
        "created_utc": c.get("created_utc"),
        "permalink": c.get("permalink"),
        "edited": _normalize_edited(c.get("edited")),
        "distinguished": _normalize_distinguished(c.get("distinguished")),
        "gilded": _safe_int(c.get("gilded"), default=0),
        "is_submitter": bool(c.get("is_submitter")),
        "stickied": bool(c.get("stickied")),
        "locked": bool(c.get("locked")),
        "collapsed": bool(c.get("collapsed")),
    }


# --------------------------------
# MAIN EXTRACTION FUNCTION
# --------------------------------

def extract_all_comments_from_json(
    thread_json: Any,
    include_more_placeholders: bool = False,  # if True, keeps "more" nodes as blank rows
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:

    post, comments_listing = _extract_post_and_top_listing(thread_json)

    comments_flat: List[Dict[str, Any]] = []
    queue: List[Tuple[Dict[str, Any], int]] = []

    for child in comments_listing.get("data", {}).get("children", []):
        queue.append((child, 0))

    while queue:
        node, depth = queue.pop(0)
        kind = node.get("kind")
        data = node.get("data", {})

        if kind == "t1":
            comments_flat.append(_flatten_comment(data, post, depth))

            replies = data.get("replies")
            if isinstance(replies, dict):
                for ch in replies.get("data", {}).get("children", []):
                    queue.append((ch, depth + 1))

        elif kind == "more":
            if include_more_placeholders:
                comments_flat.append({
                    "post_id": post.get("id"),
                    "link_fullname": post.get("name"),
                    "comment_id": None,
                    "comment_fullname": None,
                    "parent_fullname": data.get("parent_id"),
                    "depth": depth,
                    "author": None,
                    "body": None,
                    "score": 0,
                    "created_utc": None,
                    "permalink": None,
                    "edited": None,
                    "distinguished": None,
                    "gilded": 0,
                    "is_submitter": False,
                    "stickied": False,
                    "locked": False,
                    "collapsed": False,
                })
            continue

    return post, comments_flat





In [25]:
import os
import csv
import json
from typing import Any, Dict, List

FIELDNAMES = [
    "post_title","subreddit","post_id","link_fullname","comment_id","comment_fullname","parent_fullname","depth",
    "author","body","score","created_utc","permalink","edited","distinguished","gilded",
    "is_submitter","stickied","locked","collapsed","home_team","away_team"
]

def append_csv(rows: List[Dict[str, Any]], path: str) -> None:
    if not rows:
        return

    file_exists = os.path.exists(path) and os.path.getsize(path) > 0

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")

        if not file_exists:
            writer.writeheader()

        writer.writerows(rows)


def save_jsonl(rows: List[Dict[str, Any]], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def save_csv(rows: List[Dict[str, Any]], path: str) -> None:
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

# --------------------------------
# USAGE
# --------------------------------
MASTER_FILE = "all_comments_all_teams.csv"

for i in range(101,len(reddit_urls)):
    row = reddit_urls.iloc[i]
    url = str(row["Reddit link"]).strip()
    home_team = row["Home Team"]
    away_team = row["Away Team"]

    thread_json = http_get_json(url)  # make sure this returns full reddit JSON
    post_data, comments = extract_all_comments_from_json(thread_json)

    print("Post title:", post_data.get("title"))
    print("Post id:", post_data.get("id"))
    print("Total comments extracted:", len(comments))
    print("Sample comment:", comments[0] if comments else None)

    post_id = post_data.get("id")
    subreddit = post_data.get("subreddit")
    title = post_data.get("title")

    for c in comments:
        c["post_id"] = post_id
        c["subreddit"] = subreddit
        c["post_title"] = title
        c["source_url"] = url  # optional
        c["home_team"] = home_team
        c["away_team"] = away_team

    append_csv(comments, MASTER_FILE)


Post title: Post Game Thread: Carolina Panthers at Tampa Bay Buccaneers
Post id: 1q3aysk
Total comments extracted: 200
Sample comment: {'post_id': '1q3aysk', 'link_fullname': 't3_1q3aysk', 'comment_id': 'nxjce0g', 'comment_fullname': 't1_nxjce0g', 'parent_fullname': 't3_1q3aysk', 'depth': 0, 'author': 'Palifaith', 'body': 'Todd Bowles lives to see another day.', 'score': 288, 'created_utc': 1767486246.0, 'permalink': '/r/nfl/comments/1q3aysk/post_game_thread_carolina_panthers_at_tampa_bay/nxjce0g/', 'edited': None, 'distinguished': None, 'gilded': 0, 'is_submitter': False, 'stickied': False, 'locked': False, 'collapsed': False}
Post title: Post Game Thread: San Francisco 49ers at Seattle Seahawks
Post id: 1nb7p69
Total comments extracted: 198
Sample comment: {'post_id': '1nb7p69', 'link_fullname': 't3_1nb7p69', 'comment_id': 'nczppve', 'comment_fullname': 't1_nczppve', 'parent_fullname': 't3_1nb7p69', 'depth': 0, 'author': 'Mozicon', 'body': 'Despite our best efforts to blow the game, 

In [26]:
import pandas as pd

df1 = pd.read_csv("all_comments.csv")
df2 = pd.read_csv("all_comments_all_teams.csv")

stacked = pd.concat([df1, df2], ignore_index=True)

stacked.to_csv("combined.csv", index=False)